# D1-0b: a forecasting objective for sensing allocation on the Server Machine Dataset (validation machines only; the test machines are never downloaded)

Redesign of the D1-0 objective. The question, the domain card changes, every frozen choice, the gates and the decision rules are in `docs/plans/d1-0b-plan.md`; read that first. D1-0 scored allocations by fidelity to a frozen detector whose labelled F1 did not move with sensing. **Here the loss is the native endpoint: the squared error of a frozen linear forecaster's 5-minute-ahead prediction of all 38 channels against the true future.** This notebook is an exploratory Rung-0 study with non-learned policies and makes **no statement about H2**; nothing in it is a co-state. It runs on CPU; the data (MIT, register id `smd`) are fetched from `raw.githubusercontent.com` if the D1-0 cache is absent and stay out of git.

**Design in one paragraph.** A window is 60 minutes of a 38-channel stream with a snapshot of every channel at its start; the allocator opens `k` channels (observed at every minute) and the rest are held at the snapshot. Machines are split by index mod 4 into tuning (7), validation (7) and test (14, never downloaded). **Stage A (tuning machines):** correctness checks, the endpoint-moves statistic `r` (does holding instead of observing raise the forecast error by at least 5 %?) and the best fixed policy; **if `r` < 0.05 on the tuning machines the validation machines are not read.** **Stage B (validation machines):** everything is frozen and hashed, then G0 (the endpoint moves), G1 (the oracle's headroom over the best fixed allocation, at least 15 % of the normalised excess area), G2 (the share kept by the dynamic policies) and the value-of-information comparisons are computed, for the primary setting (`h` = 5, `L` = 60) and four pre-registered sensitivities (`h` = 1, 15; `L` = 30, 120).

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import sensor_forecast  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D1-0b forecasting module; set REPO_REF to a commit that has it.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
    DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/data/smd')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
    DATA_ROOT = Path.home() / 'adjointrwm_data' / 'smd'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT, '| data ->', DATA_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration, the immutable run directory and the tuning machines
# ================================================================
import datetime
import json
import platform
import tempfile
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import sensor as sn, sensor_forecast as sf
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D10bConfig:
    primary_horizon: int = 5
    primary_length: int = 60
    sensitivity_settings: tuple = ((1, 60), (15, 60), (5, 30), (5, 120))     # (horizon, window length)
    budgets: tuple = sn.DEFAULT_BUDGETS
    floor: float = sf.FLOOR
    ridge: float = sf.RIDGE
    min_endpoint_move: float = sf.MIN_ENDPOINT_MOVE
    min_relative_headroom: float = sf.MIN_HEADROOM
    bootstrap_resamples: int = 10000
    bootstrap_bank: int = 20000
    bootstrap_seed: int = 0


CFG = D10bConfig()
CONFIG_DICT = asdict(CFG)
TUNING_MACHINES = sn.split_machines('tuning')
VALIDATION_MACHINES = sn.split_machines('validation')
SPLIT_TABLE = {split: list(sn.split_machines(split)) for split in sn.SPLITS}
SETTINGS = [(CFG.primary_horizon, CFG.primary_length)] + [tuple(s) for s in CFG.sensitivity_settings]
CONFIG_HASH = sha256_json({'config': CONFIG_DICT, 'splits': SPLIT_TABLE})
RUN_ID = 'd1_0b_forecast_sensing_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

TUNING_ROWS = [row for machine in TUNING_MACHINES for row in sn.download_machine(DATA_ROOT, machine)]
atomic_write_json(PATHS['config'] / 'data_manifest_tuning.json', {'files': TUNING_ROWS, 'sha256': sha256_json(TUNING_ROWS)})
TUNING = {machine: sn.load_machine(DATA_ROOT, machine) for machine in TUNING_MACHINES}

atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'splits': SPLIT_TABLE, 'config_sha256': CONFIG_HASH,
    'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d1_0b_forecast_sensing.ipynb',
    'plan': 'docs/plans/d1-0b-plan.md', 'test_machines_downloaded': False,
})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (no accelerator needed)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12], '|', len(TUNING), 'tuning machines,', sum(r['bytes'] for r in TUNING_ROWS) // 1_000_000, 'MB')

In [ ]:
# ================================================================
# 2. Stage A on the tuning machines: the endpoint-moves statistic and the best fixed policy at every setting
# ================================================================
def fit_all(machines, horizon, length):
    return {m: sf.fit_forecaster(d.train, horizon, length, CFG.floor, CFG.ridge) for m, d in machines.items()}


STAGE_A, BEST_FIXED = {}, {}
for horizon, length in SETTINGS:
    fcs = fit_all(TUNING, horizon, length)
    results = {m: sf.evaluate_machine(fcs[m], TUNING[m].test, m, length, CFG.budgets, with_oracle=False) for m in TUNING}
    chosen = sf.choose_best_fixed(results)
    move = sf.endpoint_moves(results)
    BEST_FIXED[(horizon, length)] = chosen['selected']
    STAGE_A[f'h{horizon}_L{length}'] = {'endpoint_moves_mean': move['mean'], 'endpoint_moves_per_machine': move['per_machine'],
                                        'fixed_areas': chosen['areas'], 'best_fixed': chosen['selected'],
                                        'windows': {m: r.n_windows for m, r in results.items()}}
G0_TUNING = bool(STAGE_A[f'h{CFG.primary_horizon}_L{CFG.primary_length}']['endpoint_moves_mean'] >= CFG.min_endpoint_move)
atomic_write_json(PATHS['artifacts'] / 'tuning_summary.json', {'settings': STAGE_A, 'G0_tuning_primary': G0_TUNING, 'min_endpoint_move': CFG.min_endpoint_move})
print(pd.DataFrame({k: {'r (mean over machines)': v['endpoint_moves_mean'], 'best fixed': v['best_fixed']} for k, v in STAGE_A.items()}).T.to_string())
print('G0 on the tuning machines at the primary setting:', G0_TUNING)

In [ ]:
# ================================================================
# 3. Correctness checks on real tuning windows (G3), before anything from validation exists
# ================================================================
CORRECTNESS = {}
machine0 = TUNING_MACHINES[0]
fc0 = fit_all(TUNING, CFG.primary_horizon, CFG.primary_length)[machine0]
wins0 = sn.make_windows(fc0.standardise(TUNING[machine0].test), np.zeros(len(TUNING[machine0].test)), CFG.primary_length)
n_win, n_ch = len(wins0.index), wins0.z.shape[2]
h = CFG.primary_horizon

parts = {s: set(sn.split_machines(s)) for s in sn.SPLITS}
CORRECTNESS['split_is_a_partition'] = bool(set.union(*parts.values()) == set(sn.machine_ids()) and sum(len(p) for p in parts.values()) == len(sn.machine_ids())
                                           and [len(parts[s]) for s in sn.SPLITS] == [7, 7, 14])
spy = []
try:
    with tempfile.TemporaryDirectory() as tmp:
        sn.download_machine(tmp, sn.split_machines('test')[0], fetch=lambda url: spy.append(url) or b'')
    CORRECTNESS['fetch_refuses_a_test_machine'] = False
except PermissionError:
    CORRECTNESS['fetch_refuses_a_test_machine'] = spy == []

train = TUNING[machine0].train
z_train = (train - train.mean(axis=0)) / np.maximum(train.std(axis=0), CFG.floor)
reference_weights = np.linalg.solve(z_train[:-h].T @ z_train[:-h] + CFG.ridge * np.eye(n_ch), z_train[:-h].T @ z_train[h:])
CORRECTNESS['forecaster_reproduces_the_ridge_solution'] = bool(np.allclose(fc0.weights, reference_weights))
full = sf.native_loss(fc0, wins0, wins0.z)
direct = [np.mean([np.mean((wins0.z[w, t] @ fc0.weights - wins0.z[w, t + h]) ** 2) for t in range(1, CFG.primary_length - h)]) for w in range(min(20, n_win))]
CORRECTNESS['native_loss_at_full_observation_equals_a_direct_computation'] = bool(np.allclose(full[:len(direct)], direct))
CORRECTNESS['fidelity_is_zero_at_full_observation'] = bool(np.allclose(sf.fidelity_loss(fc0, wins0, wins0.z), 0.0))

worst = {}
for direction in ('forward', 'backward'):
    j_by_size, order = sf.oracle_greedy(fc0, wins0, direction)
    gaps = [float(np.abs(j_by_size[:, size] - sf.fidelity_loss(fc0, wins0, sn.observed(wins0.z, sn.oracle_mask(order, size, direction)))).max()) for size in range(n_ch + 1)]
    worst[direction] = max(gaps)
CORRECTNESS['fidelity_incremental_update_max_abs_difference'] = worst
CORRECTNESS['fidelity_incremental_update_matches_direct'] = bool(max(worst.values()) < 1e-8)

perturbed_z = wins0.z.copy()
perturbed_z[:, 1:, :] = np.random.default_rng(0).normal(size=perturbed_z[:, 1:, :].shape) * 100
perturbed = sn.Windows(perturbed_z, wins0.prev1, wins0.prev2, wins0.labels, wins0.index, wins0.length)
CORRECTNESS['deployable_policies_ignore_the_window_after_the_snapshot'] = bool(all(
    np.array_equal(sf.policy_mask(name, k, fc0, wins0), sf.policy_mask(name, k, fc0, perturbed)) for name in sf.DEPLOYABLE_POLICIES for k in (1, 4, 16)))
excess_full = [float(np.abs(sf.native_loss(fc0, wins0, sn.observed(wins0.z, sf.policy_mask(name, n_ch, fc0, wins0))) - full).max()) for name in sf.DEPLOYABLE_POLICIES]
CORRECTNESS['budget_38_is_full_observation_for_every_policy'] = bool(max(excess_full) < 1e-12)
CORRECTNESS['machine_used'] = machine0
CORRECTNESS['passed'] = bool(all(v for k, v in CORRECTNESS.items() if isinstance(v, bool)))
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
print(json.dumps(CORRECTNESS, indent=2))
assert CORRECTNESS['passed'], 'G3 failed: do not read validation until this is fixed'

In [ ]:
# ================================================================
# 4. Freeze everything chosen on the tuning machines; fetch the validation machines only if G0 held on the tuning machines
# ================================================================
STAGE_B = G0_TUNING
FROZEN = {'config_sha256': CONFIG_HASH, 'tuning_manifest_sha256': sha256_json(TUNING_ROWS),
          'forecaster': {'floor': CFG.floor, 'ridge': CFG.ridge, 'primary_horizon': CFG.primary_horizon},
          'best_fixed': {f'h{h_}_L{l_}': v for (h_, l_), v in BEST_FIXED.items()}, 'G0_tuning_primary': G0_TUNING, 'stage_b_run': STAGE_B,
          'rules': {'G0': f'endpoint-moves statistic r >= {CFG.min_endpoint_move} (point estimate); robust if the 95 % interval lower bound is too',
                    'G1': f'relative headroom of the oracle over the best fixed allocation >= {CFG.min_relative_headroom}; vacuous if G0 fails',
                    'G2': 'descriptive: retained fractions', 'G-VOI': 'descriptive: paired area differences', 'G3': 'correctness on real tuning windows'},
          'written_utc': datetime.datetime.now(datetime.UTC).isoformat()}
FROZEN['sha256'] = sha256_json({k: v for k, v in FROZEN.items() if k != 'written_utc'})
atomic_write_json(PATHS['reports'] / 'frozen_before_validation.json', FROZEN)
print('frozen', FROZEN['sha256'][:12], '| stage B:', STAGE_B)

if STAGE_B:
    VALIDATION_ROWS = [row for machine in VALIDATION_MACHINES for row in sn.download_machine(DATA_ROOT, machine)]
    atomic_write_json(PATHS['config'] / 'data_manifest_validation.json', {'files': VALIDATION_ROWS, 'sha256': sha256_json(VALIDATION_ROWS)})
    VALIDATION = {machine: sn.load_machine(DATA_ROOT, machine) for machine in VALIDATION_MACHINES}
    present_test = [m for m in sn.split_machines('test') if any(sn.machine_path(DATA_ROOT, kind, m).exists() for kind in sn.FILE_KINDS)]
    assert not present_test, f'test machines found in the data root: {present_test}'
    print(len(VALIDATION), 'validation machines,', sum(r['bytes'] for r in VALIDATION_ROWS) // 1_000_000, 'MB')
else:
    print('G0 failed on the tuning machines: the validation machines are not downloaded or read.')

In [ ]:
# ================================================================
# 5. Validation machines: the primary setting and the four sensitivities
# ================================================================
RESULTS, TIMINGS = {}, {}
if STAGE_B:
    for horizon, length in SETTINGS:
        key = f'h{horizon}_L{length}'
        started = time.time()
        fcs = fit_all(VALIDATION, horizon, length)
        RESULTS[key] = {m: sf.evaluate_machine(fcs[m], VALIDATION[m].test, m, length, CFG.budgets) for m in VALIDATION}
        TIMINGS[key] = round(time.time() - started, 1)
        if (horizon, length) == (CFG.primary_horizon, CFG.primary_length):
            frames = []
            for m, r in RESULTS[key].items():
                window_index = sn.make_windows(fcs[m].standardise(VALIDATION[m].test), np.zeros(len(VALIDATION[m].test)), length).index
                for name in r.native:
                    frames.append(pd.DataFrame({'machine': m, 'window': np.repeat(window_index, len(r.budgets)), 'policy': name,
                                                'budget': np.tile(np.array(r.budgets), len(window_index)), 'native_loss': r.native[name].ravel(),
                                                'fidelity_loss': r.fidelity[name].ravel(), 'full_observation_loss': np.repeat(r.full, len(r.budgets))}))
            pd.concat(frames).to_parquet(PATHS['artifacts'] / 'validation_window_losses_primary.parquet', index=False)
        print(key, '|', TIMINGS[key], 's |', {m: r.n_windows for m, r in RESULTS[key].items()})

In [ ]:
# ================================================================
# 6. Gates G0, G1, G2, G-VOI, sensitivities, table, figure and report
# ================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def fidelity_areas(results):
    """Mean over machines of each policy's normalised fidelity area (the oracle's own objective; descriptive)."""
    per = []
    for r in results.values():
        hold = float(r.fidelity['round_robin'][:, 0].mean())
        per.append({name: sn.normalised_area(table.mean(axis=0) / hold, r.budgets) for name, table in r.fidelity.items()})
    return {name: float(np.mean([p_[name] for p_ in per])) for name in per[0]}


SUMMARY = {}
if STAGE_B:
    for horizon, length in SETTINGS:
        key = f'h{horizon}_L{length}'
        summary = sf.gate_summary(RESULTS[key], BEST_FIXED[(horizon, length)], min_endpoint=CFG.min_endpoint_move, min_headroom=CFG.min_relative_headroom,
                                  num_resamples=CFG.bootstrap_resamples, seed=CFG.bootstrap_seed, bank=CFG.bootstrap_bank)
        summary.update({'horizon': horizon, 'length': length, 'budgets': list(CFG.budgets),
                        'machines': {m: {'windows': r.n_windows} for m, r in RESULTS[key].items()},
                        'fidelity_areas': fidelity_areas(RESULTS[key])})
        SUMMARY[key] = summary
    atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {'runs': SUMMARY, 'wall_clock_seconds': TIMINGS, 'validation_machines': list(VALIDATION_MACHINES)})
    PHASE = pd.DataFrame([{'setting': k, 'horizon': s['horizon'], 'length': s['length'], 'best_fixed': s['fixed'], 'endpoint_moves': s['endpoint_moves']['mean'],
                           'endpoint_ci_low': s['bootstrap']['endpoint_moves_ci'][0], 'endpoint_ci_high': s['bootstrap']['endpoint_moves_ci'][1],
                           'G0': s['G0_passes'], 'G0_robust': s['G0_robust'], 'area_fixed': s['areas'][s['fixed']], 'area_oracle': s['areas'][sf.REFERENCE],
                           'relative_headroom': s['relative_headroom'], 'ci_low': s['bootstrap']['headroom_ci'][0], 'ci_high': s['bootstrap']['headroom_ci'][1],
                           'G1': s['G1_passes'], 'G1_robust': s['G1_robust'], 'G1_vacuous': s['G1_vacuous']} for k, s in SUMMARY.items()])
    PHASE.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)
    primary = SUMMARY[f'h{CFG.primary_horizon}_L{CFG.primary_length}']
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    curves = {m: sf.machine_curves(r) for m, r in RESULTS[f'h{CFG.primary_horizon}_L{CFG.primary_length}'].items()}
    for name in list(sf.DEPLOYABLE_POLICIES) + [sf.ORACLE]:
        mean_curve = np.mean([curves[m][name] for m in curves], axis=0)
        ax.plot(CFG.budgets, mean_curve, marker='o', ms=3, lw=2 if name in (primary['fixed'], sf.ORACLE) else 1.1, ls='--' if name == sf.ORACLE else '-', label=name)
    ax.set_xlabel('channels opened per window (k)')
    ax.set_ylabel('excess forecast error over full observation / full-observation error')
    ax.set_title(f'{RUN_ID}: validation machines, h = {CFG.primary_horizon}, L = {CFG.primary_length}', fontsize=9)
    ax.legend(fontsize=7)
    fig.savefig(PATHS['figures'] / 'loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D1-0b (exploratory Rung-0 redesign)', 'config_hash': CONFIG_HASH, 'frozen_sha256': FROZEN['sha256'], 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'forecaster': FROZEN['forecaster'], 'best_fixed': FROZEN['best_fixed'], 'stage_a': STAGE_A, 'G0_tuning_primary': G0_TUNING,
    'stage_b_run': STAGE_B, 'test_machines_downloaded': False, 'validation_machines': list(VALIDATION_MACHINES) if STAGE_B else [],
    'by_setting': {k: {kk: s[kk] for kk in ('fixed', 'G0_passes', 'G0_robust', 'relative_headroom', 'G1_passes', 'G1_robust', 'G1_vacuous')}
                   | {'endpoint_moves': s['endpoint_moves']['mean'], 'endpoint_ci': s['bootstrap']['endpoint_moves_ci'], 'headroom_ci': s['bootstrap']['headroom_ci']}
                   for k, s in SUMMARY.items()},
    'claim_boundary': ('Exploratory Rung-0 study on SMD machines with non-learned policies and a frozen linear forecaster. The oracle is a privileged greedy selection '
                       'that sees the window states but not the forecast targets. No learned critic and no co-state are involved, so no H2 statement follows. '
                       'Policy computation is not charged.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D1-0b `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · G0 on the tuning machines (primary): **{'PASS' if G0_TUNING else 'FAIL'}** "
         f"({STAGE_A[f'h{CFG.primary_horizon}_L{CFG.primary_length}']['endpoint_moves_mean']:.3f}) · stage B (validation) run: **{STAGE_B}**"]
if STAGE_B:
    p = SUMMARY[f'h{CFG.primary_horizon}_L{CFG.primary_length}']
    lines += [f"G0 on validation: **{'PASS' if p['G0_passes'] else 'FAIL'}**{' (robust)' if p['G0_robust'] else ' (not robust)'} r = {p['endpoint_moves']['mean']:.3f} "
              f"[{p['bootstrap']['endpoint_moves_ci'][0]:.3f}, {p['bootstrap']['endpoint_moves_ci'][1]:.3f}] · G1: **{'PASS' if p['G1_passes'] else 'FAIL'}**"
              f"{' (robust)' if p['G1_robust'] else ' (not robust)'} headroom {p['relative_headroom']:.3f} [{p['bootstrap']['headroom_ci'][0]:.3f}, {p['bootstrap']['headroom_ci'][1]:.3f}]", '',
              '| Setting | Best fixed | r [95 % CI] | G0 | Relative headroom [95 % CI] | G1 |', '|---|---|---|---|---|---|']
    for _, r in PHASE.iterrows():
        lines.append(f"| {r['setting']} | {r['best_fixed']} | {r['endpoint_moves']:.3f} [{r['endpoint_ci_low']:.3f}, {r['endpoint_ci_high']:.3f}] | {r['G0']} | "
                     f"{r['relative_headroom']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}] | {r['G1']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

- `reports/run_summary.md` and `reports/acceptance_report.json` give G0 and G1 for every setting; **the primary setting (`h` = 5, `L` = 60) is the one the decision rules are applied to.**
- G0 says whether the forecast error rises when channels are held instead of observed; G1 is vacuous if it does not. The oracle is privileged (it sees the window states), so G1 is an upper bound on what sensing could give and G2 (how much a deployable policy keeps) and the value-of-information comparisons in `artifacts/validation_summary.json` carry the information.
- A pass licenses a D1-1 design (and needs Roman's go-ahead); a G0 failure means D1 has no sensing opportunity for forecasting on SMD at this horizon. Neither is a statement about H2.